In [4]:
import numpy as np
import matplotlib.pyplot as plt
import numba
from numba import njit, prange

In [1]:
def simulate_GBM(S0, mu,sigma, T, n_steps, n_paths):
    dt = T/n_steps
    t = np.linspace(0,T,n_steps+1)
    Z = np.random.normal(0,1.0,size=(n_paths, n_steps))
    W = np.cumsum(np.sqrt(dt)*Z, axis=1)
    W = np.hstack((np.zeros((n_paths, 1)), W))
    drift = (mu - (0.5*sigma**2))*t
    shock = sigma*W
    S = S0*np.exp(drift + shock)
    return t, S

In [2]:
def simulate_GBM_antithetic(S0, mu, sigma, T, n, n_paths):
    assert n_paths % 2 ==0
    half = n_paths//2
    dt = T/n
    t = np.linspace(0,T, n+1)
    Z=np.random.normal(0,1.0,size=(half, n))
    Z_full = np.vstack([Z,-Z])
    W = np.cumsum(np.sqrt(dt)*Z_full, axis=1)
    W=np.hstack((np.zeros((n_paths,1)),W))
    drift = (mu-0.5*sigma**2)*t
    shock = sigma*W
    S =S0*np.exp(drift+shock)
    return t,S

In [5]:

@njit
def simulate_GBM_numba(S0, mu, sigma, T, n_steps, n_paths):
    dt = T / n_steps
    sqrt_dt = np.sqrt(dt)
    S = np.empty((n_paths, n_steps + 1))
    S[:, 0] = S0
    
    for i in range(n_paths):
        s = S0
        for step in range(n_steps):
            z = np.random.normal(0.0, 1.0)
            s = s * np.exp((mu - 0.5*sigma**2)*dt + sigma*sqrt_dt*z)
            S[i, step+1] = s
    
    return S

In [6]:
@njit(parallel=True)
def simulate_GBM_numba_parallel(S0, mu, sigma, T, n_steps, n_paths):
    dt = T / n_steps
    sqrt_dt = np.sqrt(dt)
    S = np.empty((n_paths, n_steps + 1))
    S[:, 0] = S0
    
    for i in prange(n_paths):
        s = S0
        for step in range(n_steps):
            z = np.random.normal(0.0, 1.0)
            s = s * np.exp((mu - 0.5*sigma**2)*dt + sigma*sqrt_dt*z)
            S[i, step+1] = s
    
    return S